In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

p = Path.cwd()
while not (p / "pyproject.toml").exists():
    if p.parent == p:
        raise RuntimeError("project root not found")
    p = p.parent
os.chdir(p)

In [ ]:
from seizure_onset_detector.data import list_patients, list_recordings, load_info, load_recording


# Exploratory Data Analysis

- Number of patients.
- Seizures per patient.
- Total ictal vs interictal duration.
- Sampling rate consistency.
- Channel count consistency.


In [ ]:
DATA_DIR = "datasets/swec-ethz"
patients = list_patients(DATA_DIR)
print(f"Number of patients: {len(patients)}")

# Initialize lists to store patient data
seizures_all = []
fs_all = []
ictal_duration_hours_all = []
total_duration_hours_all = []
ictal_duration_hour_per_seizure_all = []
n_channels_all = []

for patient in patients:
   fs, seizure_intervals = load_info(DATA_DIR, patient)

   # Get length of ictal duration
   ictal_duration_hours = 0
   for start, end in seizure_intervals:
      assert end > start, f"Invalid seizure interval for patient {patient}: start {start} is not less than end {end}"
      ictal_duration_hour_per_seizure = (end - start) / 3600
      ictal_duration_hours += ictal_duration_hour_per_seizure
      ictal_duration_hour_per_seizure_all.append(ictal_duration_hour_per_seizure)
   ictal_duration_hours_all.append(ictal_duration_hours)

   n_seizures = len(seizure_intervals)
   seizures_all.append(n_seizures)

   fs_all.append(fs)
   recordings = list_recordings(DATA_DIR, patient)

   total_duration_hours = 0
   n_channels = 0
   for i, recording in enumerate(recordings):
      signal = load_recording(DATA_DIR, patient, recordings[i])
      total_duration_hours += signal.shape[1] / fs / 3600

      # Check consistency of number of channels across recordings for this patient
      if i == 0:
         n_channels = signal.shape[0]
      else:
         assert signal.shape[0] == n_channels, f"Patient {patient} has inconsistent number of channels across recordings: expected {n_channels}, got {signal.shape[0]}"

   n_channels_all.append(n_channels)
   total_duration_hours_all.append(total_duration_hours)

df = pd.DataFrame({
   "Patient": [f"ID{patient}" for patient in patients],
   "Seizures": seizures_all,
   "Sampling Rate (Hz)": fs_all,
   "Channels": n_channels_all,
   "Downloaded (h)": [int(hour) for hour in total_duration_hours_all],
   "Ictal (%)": [(ictal/total)*100 for ictal, total in zip(ictal_duration_hours_all, total_duration_hours_all)]
})
df.to_csv("figures/cohort_summary.csv", index=False)
print(df.to_markdown(index=False, floatfmt=".1f"))

# Composition of downloaded hours: seizure vs interictal

In [ ]:

from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(5, 2, figsize=(14, 18), sharey=True)
axes = axes.flatten()

for ax, patient in zip(axes, patients):
    _, seizure_intervals = load_info(DATA_DIR, patient)
    recordings = list_recordings(DATA_DIR, patient)

    counts = []
    for rec in recordings: 
        n = int(rec[:-1])
        rec_start, rec_end = (n - 1) * 3600, n * 3600
        counts.append(sum(
            1 for sz_start, sz_end in seizure_intervals
            if sz_start < rec_end and sz_end > rec_start
        ))

    ax.bar(recordings, counts, color="steelblue")
    ax.set_title(f"ID{patient}  ({len(seizure_intervals)} seizures total)")
    ax.set_ylabel("seizures in file")
    ax.tick_params(axis="x", rotation=45)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

sns.despine(fig=fig)
plt.tight_layout()
fig.savefig("figures/seizures_per_recording.tmp.png", dpi=150, bbox_inches="tight")
plt.show()


# Per patient seizure count bar chart

In [ ]:

fig, ax = plt.subplots()
plt.bar(df["Patient"], df["Seizures"])
median_seizures = df["Seizures"].median()
ax.axhline(median_seizures, color="crimson", linestyle="--", linewidth=1.5, label=f"Median = {median_seizures:.1f}")
ax.legend()
plt.xlabel("Patient ID")
plt.ylabel("Number of Seizures")
plt.title("Seizures per Patient")
sns.despine()
fig.savefig("figures/seizures_per_patient.png", dpi=150, bbox_inches="tight")
plt.show()

# Seizure duration histogram

In [ ]:

fig, ax = plt.subplots(figsize=(8, 6))
plt.hist([duration * 60 for duration in ictal_duration_hour_per_seizure_all], bins=15)
plt.xlabel("Seizure Duration (minutes)")
plt.ylabel("Frequency")
plt.title("Histogram of Seizure Durations")
sns.despine()
fig.savefig("figures/seizure_duration_histogram.png", dpi=150, bbox_inches="tight")
plt.show()


# Plot a sample seizure with 10s buffer on either side

In [ ]:
DATA_DIR = "datasets/swec-ethz"
RECORDING = "84h"
patient = patients[0]
fs, seizure_intervals = load_info(DATA_DIR, patient)
seizure_start, seizure_end = seizure_intervals[0]
seizure_dur_sec = seizure_end - seizure_start
print(f"Sample seizure: start={seizure_start}s, end={seizure_end}s, duration={seizure_dur_sec:.1f}s")
signal = load_recording(DATA_DIR, patient, RECORDING)
signal_subset = signal[:4, :]  # plot only first 4 channels for visibility
rec_start_time = (int(RECORDING[:-1]) - 1) * 3600  # file "Nh" covers [(N-1)*3600, N*3600)
start_idx = int(np.floor((seizure_start - rec_start_time - 10) * fs)) # add 10s buffer before seizure start
end_idx = int(np.ceil((seizure_end - rec_start_time + 10) * fs)) # add 10s buffer after seizure end

n_chans = signal_subset.shape[0]
fig, ax = plt.subplots(n_chans, 1, figsize=(12, 2*n_chans), sharex=True, constrained_layout=True)
time_axis = np.arange(start_idx, end_idx) / fs + rec_start_time  # absolute seconds
for chan in range(n_chans):
    ax[chan].plot(time_axis, signal_subset[chan, start_idx:end_idx])
    ax[chan].set_ylabel(f"Ch {chan+1}")

# Overlay every seizure that falls in the visible time range
labelled = False
for chan_ax in ax:
    for sz_start, sz_end in seizure_intervals:
        if sz_end > time_axis[0] and sz_start < time_axis[-1]:
            chan_ax.axvspan(sz_start, sz_end, alpha=0.3, color="crimson",
                            label="Seizure" if not labelled else None)
            labelled = True

ax[-1].set_xlabel("Time (s)")
sns.despine(fig=fig)
fig.suptitle(f"Patient ID{patient} - Seizure from {seizure_start:.1f}s to {seizure_end:.1f}s")
fig.legend(*ax[0].get_legend_handles_labels(), loc="outside lower center")
plt.show()


# Plot sample interictal segment from same patient with same total length as the seizure plot above

In [ ]:
DATA_DIR = "datasets/swec-ethz"
RECORDING = "73h"
patient = patients[0]
fs, seizure_intervals = load_info(DATA_DIR, patient)
signal = load_recording(DATA_DIR, patient, RECORDING)
rec_start_time = (int(RECORDING[:-1]) - 1) * 3600  # file "Nh" covers [(N-1)*3600, N*3600)

# Match the total span of the seizure plot above (seizure duration + 10s buffer on each side)
total_dur = seizure_dur_sec + 20
half_samples = int(np.ceil(total_dur * fs / 2))
mid_idx = signal.shape[1] // 2
start_idx = mid_idx - half_samples
end_idx = mid_idx + half_samples
seg_start = start_idx / fs + rec_start_time
seg_end = end_idx / fs + rec_start_time
print(f"Sample interictal segment: start={seg_start:.1f}s, end={seg_end:.1f}s, duration={total_dur:.1f}s")

signal_subset = signal[:4, :]  # plot only first 4 channels for visibility
n_chans = signal_subset.shape[0]
fig, ax = plt.subplots(n_chans, 1, figsize=(12, 2*n_chans), sharex=True, constrained_layout=True)
time_axis = np.arange(start_idx, end_idx) / fs + rec_start_time  # absolute seconds
for chan in range(n_chans):
    ax[chan].plot(time_axis, signal_subset[chan, start_idx:end_idx],
                  label="iEEG signal" if chan == 0 else None)
    ax[chan].set_ylabel(f"Ch {chan+1}")

# Overlay any seizure that falls in the visible time range (none expected for interictal)
labelled = False
for chan_ax in ax:
    for sz_start, sz_end in seizure_intervals:
        if sz_end > time_axis[0] and sz_start < time_axis[-1]:
            chan_ax.axvspan(sz_start, sz_end, alpha=0.3, color="crimson",
                            label="Seizure" if not labelled else None)
            labelled = True

ax[-1].set_xlabel("Time (s)")
fig.suptitle(f"Patient ID{patient} - Interictal segment from {seg_start:.1f}s to {seg_end:.1f}s ({RECORDING})")
sns.despine(fig=fig)
fig.legend(*ax[0].get_legend_handles_labels(), loc="outside lower center")
plt.show()
